In [1]:
import sys
sys.path.insert(0, '/storage/agrp/barakma/PileupODD')

In [2]:

%load_ext autoreload
%autoreload 2

In [3]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np

from huggingface_hub import HfFileSystem
import polars as pl

fs = HfFileSystem()

number_of_files = 1
event_type = 'ttbar_pu200'
number_of_hf_repo_files= 1000
# Load particles
particles_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_particles/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        particles_list.append(pl.read_parquet(f, columns=[    'event_id',
    'particle_id',
    'vertex_primary',
    'pdg_id',
    'energy',
    'px',
    'py',
    'pz',
    'vx',
    'vy',
    'vz',
    'parent_id',
]))
particles = pl.concat(particles_list)

# Load calo_hits
calo_hits_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_calo_hits/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        calo_hits_list.append(pl.read_parquet(f, columns=    ['event_id',
    'detector',
    'total_energy',
    'x',
    'y',
    'z',
    'contrib_particle_ids',
    'contrib_energies'
]))
calo_hits = pl.concat(calo_hits_list)

# Load tracks
tracks_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_tracks/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        tracks_list.append(pl.read_parquet(f))
tracks = pl.concat(tracks_list)

In [4]:
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import numpy as np

from huggingface_hub import HfFileSystem
import polars as pl

fs = HfFileSystem()

number_of_files = 1
event_type = 'pileup_only_pu0'
number_of_hf_repo_files= 1000
# Load particles
particles_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_particles/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        particles_list.append(pl.read_parquet(f, columns=[    'event_id',
    'particle_id',
    'vertex_primary',
    'pdg_id',
    'energy',
    'px',
    'py',
    'pz',
    'vx',
    'vy',
    'vz',
    'parent_id',
]))
particlespu = pl.concat(particles_list)

# Load calo_hits
calo_hits_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_calo_hits/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        calo_hits_list.append(pl.read_parquet(f, columns=    ['event_id',
    'detector',
    'total_energy',
    'x',
    'y',
    'z',
    'contrib_particle_ids',
    'contrib_energies'
]))
calo_hitspu = pl.concat(calo_hits_list)

# Load tracks
tracks_list = []
for i in range(number_of_files):
    file_path = f"datasets/CERN/ColliderML-Release-1/data/{event_type}_tracks/train-{i:05d}-of-{number_of_hf_repo_files:05d}.parquet"
    with fs.open(file_path, "rb") as f:
        tracks_list.append(pl.read_parquet(f))
trackspu = pl.concat(tracks_list)

In [5]:
calo_hits.head()

event_id,detector,total_energy,x,y,z,contrib_particle_ids,contrib_energies
u32,list[u8],list[f32],list[f32],list[f32],list[f32],list[list[u64]],list[list[f32]]
0,"[9, 10, … 12]","[0.000816, 0.000229, … 0.001104]","[202.36615, 142.800003, … -426.024139]","[296.12851, -1333.199951, … -383.597717]","[-3308.449951, 1071.0, … -3698.5]","[[140074], [176808], … [101665]]","[[0.000816], [0.000229], … [0.001104]]"
1,"[11, 11, … 10]","[0.000176, 0.000072, … 0.000057]","[-81.599998, -306.947235, … 288.261261]","[979.78894, -343.009705, … 1268.982788]","[3399.350098, 3293.300049, … 2315.399902]","[[63413], [37771], … [144414]]","[[0.000176], [0.000072], … [0.000057]]"
2,"[11, 9, … 9]","[0.000157, 0.001176, … 0.004281]","[449.388947, -1191.736206, … -588.234314]","[56.099998, 256.264709, … 458.409515]","[3389.25, -3257.949951, … -3237.75]","[[163748], [104990, 104993], … [165102, 165103, … 165130]]","[[0.000157], [0.000945, 0.00023], … [0.00082, 0.000162, … 0.000157]]"
3,"[11, 10, … 11]","[0.000953, 0.000507, … 0.000152]","[174.848709, 820.349915, … 1199.088989]","[-728.641785, 1015.087158, … -5.1]","[3222.600098, 3014.100098, … 3343.800049]","[[169741, 169742], [126776], … [59272]]","[[0.000787, 0.000166], [0.000507], … [0.000152]]"
4,"[11, 9, … 9]","[0.000799, 0.000682, … 0.000127]","[119.866661, 685.602905, … -343.009705]","[-569.249512, 736.090332, … -321.372223]","[3263.0, -3247.850098, … -3207.449951]","[[120679, 183561], [212636], … [29980]]","[[0.000629, 0.00017], [0.000682], … [0.000127]]"


In [6]:
(calo_hitspu
 .select('x', 'y', 'z')
 .explode('x', 'y', 'z')
 .filter((pl.col('x')== 202.36615) & (pl.col('y')== 296.12851) & (pl.col('z')== -3308.449951))
 )

x,y,z
f32,f32,f32
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
…,…,…
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951


In [ ]:
(calo_hitspu
 .select('x', 'y', 'z')
 .explode('x', 'y', 'z')
 .filter((pl.col('x')== 202.36615) & (pl.col('y')== 296.12851) & (pl.col('z')== -3308.449951))
 )

In [ ]:
(calo_hitspu
 .select('x', 'y', 'z')
 .explode('x', 'y', 'z')
 .
 .filter()
 )

x,y,z
f32,f32,f32
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
…,…,…
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
202.36615,296.12851,-3308.449951
